# InstructBLIP hallucination traversal

Companion to the paper's §*Attractors for traversals / hallucinations*. Qualitative single-image demo with InstructBLIP (Vicuna-7B via `salesforce-lavis`): cache the mean image-token activation at a middle attention layer, then inject a scaled copy of it on every generation step to "remind" the model of the image and suppress hallucinated content as `strength` grows.

## Setup

In [ ]:
import requests
from PIL import Image

import torch
from lavis.models import load_model_and_preprocess
import os
import matplotlib.pyplot as plt 

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Model

In [ ]:
model, vis_processors, _ = load_model_and_preprocess(name="blip2_vicuna_instruct", model_type="vicuna7b", is_eval=True, device=device)

## Image

Fetch one COCO sample over HTTP.

In [ ]:
prompt = "Describe the image in detail."

image_file = "http://images.cocodataset.org/val2017/000000039769.jpg"
raw_image = Image.open(requests.get(image_file, stream=True).raw)
inputs = vis_processors["eval"](raw_image).unsqueeze(0)


In [ ]:
plt.imshow(raw_image)
plt.show()

## Baseline caption

Caption the image before any steering intervention.

In [ ]:
model.generate({"image": inputs.to(device), "prompt": prompt}, use_nucleus_sampling=True, max_length=1024)

## Attractor intervention

Register a forward hook on `self_attn` of one middle decoder layer. On the prefill pass (multi-token input) the hook caches the mean-pooled image-token activation into `concept`. On every subsequent single-token generation step it adds `strength * concept` to the attention output — "reminding" the model of the image and suppressing the drift that produces hallucinated content.

In [ ]:
target_layer = 17

In [ ]:
concept = None

def remind_image(target_layer):
    global concept
    def hook(model, input, output):
        global concept
        if (output[0].shape[1] != 1):
            concept = torch.mean(output[0], dim=1, keepdim=True)
        else:
            output = (output[0] + strength*concept, *output[1:])
        return output
    return hook


hook = model.llm_model.model.layers[target_layer].self_attn.register_forward_hook(remind_image(target_layer))

## Steered sweep

Regenerate while sweeping `strength`. At 0 you recover the baseline; as strength grows the caption stays more grounded in what's actually in the image.

In [ ]:

for strength in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4]:
    print (strength)
    print(model.generate({"image": inputs.to(device), "prompt": prompt}, use_nucleus_sampling=True, max_length=1024))